In [1]:
import pandas as pd
#from pandas.io.parsers import ParserError
import numpy as np
from helper import get_mapper
import json
import os
import re

In [2]:
from os import listdir, stat
from os.path import isfile, join
BASE_DIR = "."
MIN_SIZE = 512

In [127]:
def extract_blockidf(fullname):
    return fullname.split("Generation_DE ")[1].rsplit('[MW]')[0].strip()

def extract_blockidf2(fullname, plantname):
    return fullname.split("Generation_DE ")[1].rsplit('[MW]')[0].strip().removeprefix(plantname + " ")

def get_smard_name(f):
    return f.rsplit("/")[2].rsplit("_", 3)[0].strip()

def get_smard_name_win(f):
    f = f.replace("\\","/", 2)
    return f.rsplit("/")[2].rsplit("_", 4)[0].strip()

In [128]:
get_smard_name_win("'.\\2015/Abwinden-Asten_201501010000_201512312359_Stunde_1.csv'")

'Abwinden-Asten'

In [129]:
get_smard_name("./2015/Abwinden-Asten_201501010000_201512312359_hour_1.csv")

'Abwinden-Asten_201501010000'

In [94]:
def get_files_from_folder(folder):
    onlyfiles = [folder + "/" + f for f in listdir(folder) if isfile(join(folder, f))]
    onlyfiles.sort()
    files = [f for f in onlyfiles if stat(f).st_size > MIN_SIZE]
    return files

In [95]:
def convert2plantid(df, plantname):
    oldcols = list(df.columns)
    newcols = [extract_blockidf2(x, plantname) for x in list(df.columns)[1:]]
    #print(plantname + ":" + str(newcols))
    try:
        newcols2 = ['produced_at'] + [mapper[plantname][x] for x in newcols]
    except KeyError:
        newcols2 = ['produced_at'] + [mapper[plantname][x.split(" ")[-1]] for x in newcols]
    test = dict(zip(oldcols, newcols2))
    result = df.rename(columns=test)
    #print(oldcols)
    #print(newcols2)
    return result

In [96]:
def get_plant_from_prod_name(prodname):
    tmp = ""
    try:
        tmp = mapper[prodname]
    except KeyError:
        #print("plant " + prodname + " not found!")
        return None, None
    blocklist = tmp['list']
    block = blocklist[0]
    blockid = tmp[block]
    
    try:
        plantidx = bpm.loc[bpm.blockid == blockid, 'plantid'].item()
    except ValueError:
        return blockid, np.nan
    
    return blockid, plantidx

In [97]:
def get_plant_from_prod_name2(prodname):
    tmp = ""
    try:
        tmp = newmapper[prodname]
    except KeyError:
        #print("plant " + prodname + " not found!")
        return None, None
    seelist = tmp['list']
    block = seelist[0]
    blockid = tmp[block]
    
    try:
        plantidx = seem.loc[seem.sseid == blockid, 'plantid'].item()
    except ValueError:
        return blockid, np.nan
    
    return blockid, plantidx

In [98]:
#bpm = pd.read_csv("../basic/block_plant_mapper.csv")
seem = pd.read_csv("../basic/plant_sse_mapper.csv")
#bpm['plantid'] = bpm['plantid'].apply(lambda x: str(x).replace('/', '_'))
seem['plantid'] = seem['plantid'].apply(lambda x: str(x).replace('/', '_'))

In [99]:
seem

,plantid,sseid
0,12-30532000000,SEE969511682320
1,06-05-900-0327252,SEE948351715917
2,14-80-55047380000,SEE986249505394
3,14-80-55047380000,SEE955310827123
4,12-30532000000,SEE939848582457
...,...,...
1127,03-10-10101531850,SEE908658611688
1128,03-10-10285139970,SEE940780197517
1129,06-05-300-0215443,SEE924053403383
1130,06-05-300-0215443,SEE990523934756


In [100]:
FOLDERS = [os.path.join(BASE_DIR, o) for o in os.listdir(BASE_DIR) if os.path.isdir(os.path.join(BASE_DIR,o))]
FOLDERS.sort()
FOLDERS = FOLDERS[1:-4]
#FOLDERS = FOLDERS[0:1]

In [101]:
FOLDERS

['./2025']

In [102]:
FILES_L = [get_files_from_folder(f) for f in FOLDERS]
FILES = [item for sublist in FILES_L for item in sublist]

In [103]:
#FILES

In [104]:
#mapper = get_mapper('../production/plantmapper.json')
mapper = get_mapper('newmapper.json')
newmapper = get_mapper('newmapper.json')

In [105]:
#get_plant_from_prod_name("Buschhaus")

In [106]:
get_plant_from_prod_name2("Neurath")

('SEE993592183001', '06-05-100-0248923')

In [107]:
#mapper

In [108]:
seem

,plantid,sseid
0,12-30532000000,SEE969511682320
1,06-05-900-0327252,SEE948351715917
2,14-80-55047380000,SEE986249505394
3,14-80-55047380000,SEE955310827123
4,12-30532000000,SEE939848582457
...,...,...
1127,03-10-10101531850,SEE908658611688
1128,03-10-10285139970,SEE940780197517
1129,06-05-300-0215443,SEE924053403383
1130,06-05-300-0215443,SEE990523934756


In [109]:
#FILES

In [110]:
#FILES

In [111]:
def get_year(fn):
    return int(fn.split("/")[1])

In [112]:
def get_year2(fn):
    return int(fn.rsplit("_", 3)[1][0:4])

In [113]:
get_year2("./2015/Boxberg_201501010000_201512312345_Stunde_71.csv")

2015

In [130]:
FILES[1]

'./2025/Kraftwerk_Schwarze_Pumpe_202501010000_202512192359_Viertelstunde.csv'

In [131]:
for f in FILES:
    #print(f[1])
    fn = get_smard_name(f)
    print(fn)

Kraftwerk_Neurath
Kraftwerk_Schwarze_Pumpe


In [144]:
prod_mapper = []
for f in FILES:
    fn = get_smard_name(f)
    #f = f.replace("\\","/", 2)
    #print(fn)
    blockid, plantid = get_plant_from_prod_name2(fn)
    print(fn or "" + ":" + blockid or "" + "->" + plantid or "")
    year = 2015
    try:
        year = get_year2(f)
    except IndexError:
        print(fn)
        pass
    prod_mapper.append([f, blockid, plantid, year])

Kraftwerk_Neurath
Kraftwerk_Schwarze_Pumpe


In [145]:
date_format={'Datum von': '%d-%m-%Y %H:%M'}

In [146]:
df = pd.read_csv("./2025/Kraftwerk_Neurath_202501010000_202512192359_Viertelstunde.csv", delimiter=";", na_values=['-'], thousands='.', decimal=',', parse_dates=["Datum von"], date_format={'Datum von': '%d-%m-%Y %H:%M'}, on_bad_lines='skip')

In [147]:
prod_mapper

[['./2025/Kraftwerk_Neurath_202501010000_202512192359_Viertelstunde.csv',
  None,
  None,
  2025],
 ['./2025/Kraftwerk_Schwarze_Pumpe_202501010000_202512192359_Viertelstunde.csv',
  None,
  None,
  2025]]

In [148]:
df.fillna(0, inplace=True)
df[df.columns[2:]] = df[df.columns[2:]].astype(int)

In [149]:
df[df.columns[2:]] = df[df.columns[2:]].astype(int)

In [150]:
prd_df = pd.DataFrame(prod_mapper, columns = ['file', 'blockid', 'plantid', 'year'])

In [151]:
dedup = prd_df.dropna(subset=['blockid', 'plantid'])

In [152]:
dedup

,file,blockid,plantid,year


In [153]:
for index, row in list(dedup.iterrows()):
    dfname = row.iloc[0]
    plantid = str(row.iloc[2])
    year = str(int(row.iloc[3]))
    smardname = get_smard_name(dfname)
    try:
        df = pd.read_csv(dfname, delimiter=";", na_values=['-'], thousands='.', decimal=',', parse_dates=["Datum von"], on_bad_lines='skip')
        df['Datum von'] = pd.to_datetime(df['Datum von'], format='%d.%m.%Y %H:%M')
        df = df.drop('Datum bis', axis=1)
    except ValueError:
        df = pd.read_csv(dfname, delimiter=";", na_values=['-'], thousands='.', decimal=',', parse_dates=["Datum von"], on_bad_lines='skip')
        df = df.drop('Datum bis', axis=1)
    try:
    #print(smardname)
        newdf = convert2plantid(df, smardname)
        newdf.fillna(0, inplace=True)
        newdf[newdf.columns[1:]] = newdf[newdf.columns[1:]].astype(int)
        newdf.to_csv("./by_plantid/" + year + "/" + plantid + '.csv', index=False)
    except (IndexError, KeyError, ValueError):
        print(smardname)
        continue
    
    #try:
    #print(dfname, year)
    #print(type(year))
    
    #except TypeError:
    #    print(year, smardname)
    #print(newdf.dtypes)

In [142]:
extract_blockidf('Generation_DE Bergkamen A [MW] Originalauflösungen\n')

'Bergkamen A'